In [ ]:
#we want to convert text -> numerical values
#1. we need a vocabulary that maps each word to a unique integer
#2. we need to setup a pytorch dataset to load the data 
#3. setup padding and batching
#4

In [9]:
import os
import pandas as pd 
import torch
from torch.utils.data import Dataset,DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch.nn as nn
from PIL import Image
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torchvision
import spacy
from tqdm import tqdm
spacy_eng = spacy.load("en_core_web_sm")


In [10]:
class Vocabulary:
    def __init__ (self, freq_threshold):
        self.freq_threshold = freq_threshold
        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.stoi = {v:k for k,v in self.itos.items()}
        # self.nlp = spacy.load("en_core_web_sm")

    def __len__(self):
        return len(self.itos)
    @staticmethod
    def tokenizer_eng(text):
        # Tokenize English text into words
        return [token.text.lower() for token in spacy_eng.tokenizer(text)]
    def build_vocabulary(self, sentence_list):
        frequencies = {}
        idx = 4

        for i, sentence in enumerate(sentence_list):
            if i % 500 == 0:
                print(f"Processed {i}/{len(sentence_list)} captions")
            for word in self.tokenizer_eng(sentence):
                if word not in frequencies:
                    frequencies[word] = 1
                else:
                    frequencies[word] += 1

                if frequencies[word] == self.freq_threshold:
                    self.stoi[word] = idx
                    self.itos[idx] = word
                    idx += 1
    def numericalize(self, text):
        tokenized_text = self.tokenizer_eng(text)
        return [
            self.stoi.get(token, self.stoi["<UNK>"])
            for token in tokenized_text
        ]


In [11]:
class FlickerDataset(Dataset):
    def __init__(self,root,caption_file,transform=None,freq_threshold=5):
        self.root = root
        self.transform = transform
        self.df = pd.read_csv(caption_file)
        #get images caption columns
        self.image_names = self.df['image']
        self.captions = self.df['caption']

        self.freq_threshold = freq_threshold    
        # self.nlp = spacy.load("en_core_web_sm")
        self.vocab = Vocabulary(freq_threshold)
        self.vocab.build_vocabulary(self.captions.tolist())
        # self.vocab = self.build_vocab(self.captions)
        
    def build_vocab(self,captions):
        vocab = {}
        idx = 0
        for caption in captions:
            tokens = self.nlp(caption.lower())
            for token in tokens:
                if token.text not in vocab:
                    vocab[token.text] = idx
                    idx += 1
        return vocab
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self,idx):
        img_name = os.path.join(self.root,self.image_names[idx])
        image = Image.open(img_name).convert("RGB")
        if self.transform:
            image = self.transform(image)
        
        caption = self.captions[idx]

        numerical_caption = [self.vocab.stoi["<SOS>"]]
        numerical_caption += self.vocab.numericalize(caption)
        numerical_caption.append(self.vocab.stoi["<EOS>"])

        return image , torch.tensor(numerical_caption)
        

In [12]:
class MyCollate:
    def __init__(self, pad_idx):
        self.pad_idx = pad_idx

    def __call__(self, batch):
        images = [item[0].unsqueeze(0) for item in batch]
        images = torch.cat(images, dim=0)
        captions = [item[1] for item in batch]
        captions = pad_sequence(captions, batch_first=False, padding_value=self.pad_idx)

        return images, captions

In [13]:
def get_loader(
    root,
    caption_file,
    transform,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
):
    dataset = FlickerDataset(root=root, caption_file=caption_file, transform=transform)

    pad_idx = dataset.vocab.stoi["<PAD>"]

    loader = DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=pin_memory,
        collate_fn=MyCollate(pad_idx=pad_idx),
    )
    loader = tqdm(loader, desc="Preparing Data", leave=False)
    return loader

In [17]:


def main():
    dataloader = get_loader(
        root="flickr8k/images",
        caption_file="flickr8k/captions.txt",
        transform=transforms.Compose(
            [
                transforms.Resize((256, 256)),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]
                )
            ]
        ),
        batch_size=4,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )

    # tqdm progress bar
    for idx, (imgs, captions) in enumerate(tqdm(dataloader, desc="Loading Batches")):
        print(f"Batch {idx+1}:")
        print(imgs.shape)
        print(captions.shape)
        print("captions:", captions)
        print("images:", imgs)
        break
        


if __name__ == "__main__":
    main()


Processed 0/40455 captions
Processed 500/40455 captions
Processed 1000/40455 captions
Processed 1500/40455 captions
Processed 2000/40455 captions
Processed 2500/40455 captions
Processed 3000/40455 captions
Processed 3500/40455 captions
Processed 4000/40455 captions
Processed 4500/40455 captions
Processed 5000/40455 captions
Processed 5500/40455 captions
Processed 6000/40455 captions
Processed 6500/40455 captions
Processed 7000/40455 captions
Processed 7500/40455 captions
Processed 8000/40455 captions
Processed 8500/40455 captions
Processed 9000/40455 captions
Processed 9500/40455 captions
Processed 10000/40455 captions
Processed 10500/40455 captions
Processed 11000/40455 captions
Processed 11500/40455 captions
Processed 12000/40455 captions
Processed 12500/40455 captions
Processed 13000/40455 captions
Processed 13500/40455 captions
Processed 14000/40455 captions
Processed 14500/40455 captions
Processed 15000/40455 captions
Processed 15500/40455 captions
Processed 16000/40455 captions
P

Loading Batches:   0%|          | 0/10114 [00:00<?, ?it/s]

Batch 1:
torch.Size([4, 3, 256, 256])
torch.Size([20, 4])
captions: tensor([[   1,    1,    1,    1],
        [   4,    4,    4,    4],
        [  28,    7,    9,    9],
        [   8,  316,    7,    7],
        [   4,   76,   32,   32],
        [ 195,    4,   76,   10],
        [ 151,  157,    4,  711],
        [  17,   74,  157,   27],
        [  32,    5, 2409,  104],
        [  67,    2,    5, 2409],
        [   4,    0,    2,    5],
        [ 353,    0,    0,    2],
        [  11,    0,    0,    0],
        [ 711,    0,    0,    0],
        [   8,    0,    0,    0],
        [  24,    0,    0,    0],
        [   3,    0,    0,    0],
        [ 496,    0,    0,    0],
        [   5,    0,    0,    0],
        [   2,    0,    0,    0]])
images: tensor([[[[-0.7822, -0.2342, -0.0629,  ..., -2.0494, -2.0152, -2.0323],
          [-0.8335, -0.1999, -0.0458,  ..., -1.9980, -1.9980, -1.9467],
          [-0.8678, -0.1657, -0.0458,  ..., -1.9809, -2.0152, -1.8782],
          ...,
          [ 